![image.png](https://i.imgur.com/4fN73lZ.png)

# DPO from Scratch: Preference-Tune a Tiny Language Model Without a Reward Model

The RLHF lab learned a reward model and ran an RL loop against it. **Direct Preference
Optimization (DPO)** throws away *both*: it tunes the policy **straight from the
preference pairs** with a single supervised-style loss. No reward model, no RL loop.

We do it on a **real (tiny) language model**, trained from scratch and running in
**seconds on a CPU**:

1. **A tiny language.** A ~30-word vocabulary of mini movie-reviews with a *checkable*
   property: how **positive** a review is (`#positive - #negative` words).
2. **Stage 0 -- SFT.** Train a small GPT on a neutral corpus so it writes plausible
   reviews. Freeze a copy as the reference $\pi_{\text{ref}}$.
3. **Preferences.** Build pairs where the more positive review is preferred.
4. **DPO.** Tune the policy directly on those pairs. **(TASK 1: sequence log-prob;
   TASK 2: the DPO loss.)**
5. **The payoff.** Show the *implicit* reward $\hat r(y)=\beta\log\frac{\pi_\theta(y)}
   {\pi_{\text{ref}}(y)}$ -- read straight off the tuned LM, with no reward model --
   recovers the preference ranking. **Your LM is secretly a reward model.**
   **(TASK 3.)**

Why a real LM here and a toy in the RLHF lab? DPO has **no RL loop** -- only forward
passes over a *fixed* set of pairs -- so it is the cheap, stable place to use an actual
language model.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" torch numpy matplotlib

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cpu")
torch.manual_seed(0)
np.random.seed(0)
print("device:", device)

config = {
    "n_embd": 64, "n_head": 2, "n_layer": 2,   # tiny GPT
    "corpus_size": 3000,
    "sft_steps": 600, "sft_lr": 3e-3, "sft_batch": 128,
    "num_pairs": 800,
    "beta": 0.1, "dpo_steps": 400, "dpo_lr": 1e-3, "dpo_batch": 64,
    "rm_steps": 300, "rm_lr": 1e-3,
}

## The Setup, Concretely

Before any code, here is exactly what we build.

**The language.** A vocabulary of **28 word-tokens** (6 subjects, 4 verbs, 6 positive
adjectives, 6 negative adjectives, and a few structural tokens). Every "review" is a
fixed shape of **9 tokens**:

> `<bos> the <subject> <verb>`  **`<adjective> and <adjective> . <eos>`**

- The first **4 tokens are the prompt** (given to the model).
- The last **5 tokens are the response** (what the model writes, and what we score).

**The one property we care about** is **positivity** = (number of positive adjectives)
$-$ (number of negative adjectives). With two adjectives per review it lands in
$\{-2, -1, 0, +1, +2\}$. This is a *checkable rule* -- it replaces the giant BERT
sentiment classifier used in the classic GPT-2/IMDB demo, so the lab needs nothing
external.

**The model** is a small GPT (2 layers, 2 heads) trained from scratch on this language.
It *is* the policy $\pi_\theta$: given the 4-token prompt, it puts a probability on each
possible next token and samples the 5-token response.

**What we are trying to achieve.** Start from a model that writes *neutral* reviews
(positive and negative equally likely, average positivity $\approx 0$). Using **only a
fixed set of preference pairs** -- "this review is more positive than that one" -- and
**no reward model and no RL loop**, push the model to write *positive* reviews (average
positivity toward $+2$). Then show that the tuned model has secretly become a reward
model for positivity.

**The corpus:** 3000 neutral reviews for the SFT step; the exact counts live in
`config` above.

In [ ]:
# ---- the vocabulary ----
SUBJECTS = ["movie", "film", "story", "acting", "plot", "ending"]
VERBS    = ["was", "is", "felt", "seemed"]
POSITIVE = ["great", "brilliant", "lovely", "fun", "clever", "good"]
NEGATIVE = ["bad", "awful", "boring", "dull", "weak", "poor"]
SPECIAL  = ["<pad>", "<bos>", "<eos>", "the", "and", "."]
VOCAB = SPECIAL + SUBJECTS + VERBS + POSITIVE + NEGATIVE

stoi = {word: index for index, word in enumerate(VOCAB)}   # word  -> token id
itos = {index: word for word, index in stoi.items()}       # token id -> word
POSITIVE_IDS = {stoi[word] for word in POSITIVE}
NEGATIVE_IDS = {stoi[word] for word in NEGATIVE}
PAD, BOS, EOS = stoi["<pad>"], stoi["<bos>"], stoi["<eos>"]

PROMPT_LEN = 4                       # <bos> the <subject> <verb>
RESPONSE_LEN = 5                     # <adjective> and <adjective> . <eos>
BLOCK_SIZE = PROMPT_LEN + RESPONSE_LEN


def make_review(positive_prob):
    """Build one 9-token review; positive_prob = chance each adjective is positive."""
    subject = np.random.choice(SUBJECTS)
    verb = np.random.choice(VERBS)
    adjective_1 = np.random.choice(POSITIVE if np.random.random() < positive_prob else NEGATIVE)
    adjective_2 = np.random.choice(POSITIVE if np.random.random() < positive_prob else NEGATIVE)
    prompt = [BOS, stoi["the"], stoi[subject], stoi[verb]]
    response = [stoi[adjective_1], stoi["and"], stoi[adjective_2], stoi["."], EOS]
    return prompt + response


def positivity(response_ids):
    """#positive - #negative adjectives in a response (the checkable property)."""
    return (sum(token in POSITIVE_IDS for token in response_ids)
            - sum(token in NEGATIVE_IDS for token in response_ids))


def decode(token_ids):
    """Turn a list of token ids back into words, stopping at <eos>."""
    words = []
    for token in token_ids:
        token = int(token)
        if token == PAD:
            continue
        words.append(itos[token])
        if token == EOS:
            break
    return " ".join(words)


# neutral 50/50 corpus for SFT
corpus = torch.tensor([make_review(0.5) for _ in range(config["corpus_size"])])
print("vocab size:", len(VOCAB), "| tokens per review:", BLOCK_SIZE,
      "(prompt", PROMPT_LEN, "+ response", RESPONSE_LEN, ") | corpus:", tuple(corpus.shape))
print("example review:", decode(corpus[0]))

## A Tiny GPT, and the One Primitive DPO Needs &mdash; TASK 1

The model is a minimal decoder-only Transformer (2 layers, 2 heads). The GPT itself is
**given**. The single primitive DPO is built on is the **sequence log-probability of a
response**: how much total log-probability the model assigns to the response tokens
(the prompt tokens do not count). That is **TASK 1** -- you will reuse it everywhere
below.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd))

    def forward(self, x, causal_mask):
        normed = self.ln1(x)
        attended, _ = self.attn(normed, normed, normed, attn_mask=causal_mask, need_weights=False)
        x = x + attended
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        seq_len = idx.shape[1]
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
        x = self.token_emb(idx) + self.pos_emb(torch.arange(seq_len))
        for block in self.blocks:
            x = block(x, causal_mask)
        return self.head(self.ln(x))


def sequence_logprob(model, sequences, response_start):
    """Total log-prob the model assigns to the RESPONSE tokens of each sequence."""
    logits = model(sequences[:, :-1])
    log_probs = F.log_softmax(logits, dim=-1)
    next_tokens = sequences[:, 1:]
    response_mask = torch.zeros(next_tokens.shape)
    response_mask[:, response_start - 1:] = 1.0     # 1 on response tokens, 0 on the prompt
    token_log_prob = log_probs.gather(-1, next_tokens.unsqueeze(-1)).squeeze(-1)
    return (token_log_prob * response_mask).sum(dim=1)

## Stage 0: Supervised Fine-Tuning (SFT)

Train the GPT on the **neutral** corpus (positive and negative reviews mixed 50/50) by
ordinary next-token prediction. The result writes grammatical reviews with **no
sentiment lean** -- roughly zero average positivity. We then **freeze a copy** as the
reference model $\pi_{\text{ref}}$; DPO will pull the policy away from it, and the
implicit reward will be measured relative to it.

In [ ]:
policy = TinyGPT(len(VOCAB), BLOCK_SIZE, config["n_embd"], config["n_head"], config["n_layer"])
optimizer = torch.optim.AdamW(policy.parameters(), lr=config["sft_lr"])

for step in range(config["sft_steps"]):
    batch = corpus[torch.randint(0, len(corpus), (config["sft_batch"],))]
    logits = policy(batch[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, len(VOCAB)), batch[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"SFT done | final cross-entropy {loss.item():.3f}")


@torch.no_grad()
def generate(model, num_samples):
    """Sample one response per random prompt from the model. Returns full 9-token rows."""
    subject_ids = [stoi[subject] for subject in SUBJECTS]
    verb_ids = [stoi[verb] for verb in VERBS]
    prompts = [[BOS, stoi["the"], np.random.choice(subject_ids), np.random.choice(verb_ids)]
               for _ in range(num_samples)]
    sequence = torch.tensor(prompts)
    for _ in range(RESPONSE_LEN):
        next_token_logits = model(sequence)[:, -1, :]
        next_token = torch.multinomial(F.softmax(next_token_logits, dim=-1), 1)
        sequence = torch.cat([sequence, next_token], dim=1)
    return sequence


@torch.no_grad()
def mean_positivity(model, num_samples=400):
    samples = generate(model, num_samples)
    return np.mean([positivity(samples[row, PROMPT_LEN:].tolist()) for row in range(num_samples)])


print(f"SFT model average positivity: {mean_positivity(policy):+.2f}   (neutral, near 0)")
print("sample reviews:")
for sequence in generate(policy, 5):
    print("  ", decode(sequence))

# freeze a copy as the reference model
reference = copy.deepcopy(policy)
for parameter in reference.parameters():
    parameter.requires_grad_(False)

## Build the Preference Pairs

Sample responses from the SFT model and pair them up; within each pair, the **more
positive** review is the winner $y^+$ and the other is the loser $y^-$ (ties dropped).
This is the fixed dataset DPO will train on -- no reward model, no environment.

In [ ]:
@torch.no_grad()
def build_preference_pairs(model, num_pairs):
    """Sample responses, pair them up, and let the more positive one win."""
    winners, losers = [], []
    while len(winners) < num_pairs:
        samples = generate(model, 64)
        for row in range(0, 64, 2):
            response_a, response_b = samples[row], samples[row + 1]
            positivity_a = positivity(response_a[PROMPT_LEN:].tolist())
            positivity_b = positivity(response_b[PROMPT_LEN:].tolist())
            if positivity_a == positivity_b:
                continue                       # drop ties
            if positivity_a > positivity_b:
                winners.append(response_a)
                losers.append(response_b)
            else:
                winners.append(response_b)
                losers.append(response_a)
    return torch.stack(winners[:num_pairs]), torch.stack(losers[:num_pairs])


winners, losers = build_preference_pairs(policy, config["num_pairs"])
print(f"built {len(winners)} preference pairs (winner = the more positive review)")
print("example  winner:", decode(winners[0]))
print("example  loser :", decode(losers[0]))

## DPO: One Loss, Straight on the Policy &mdash; TASK 2

Recall the DPO loss derived in the lecture -- the *same* $-\log\sigma(\Delta)$ shape as
the reward-model loss, but with an **implicit** reward read off the policy itself:
$$L_{\text{DPO}} = -\log\sigma\!\Big(\underbrace{\beta\log\tfrac{\pi_\theta(y^+)}{\pi_{\text{ref}}(y^+)}}_{\hat r(y^+)} - \underbrace{\beta\log\tfrac{\pi_\theta(y^-)}{\pi_{\text{ref}}(y^-)}}_{\hat r(y^-)}\Big).$$
In words: raise the policy's favouring of $y^+$ over $y^-$, measured *relative to the
reference*. No reward model is trained; no responses are sampled during the update.
**TASK 2** is this loss.

In [ ]:
def dpo_loss(policy, reference, win_batch, lose_batch, beta):
    policy_logp_win  = sequence_logprob(policy, win_batch, PROMPT_LEN)
    policy_logp_lose = sequence_logprob(policy, lose_batch, PROMPT_LEN)
    with torch.no_grad():
        ref_logp_win  = sequence_logprob(reference, win_batch, PROMPT_LEN)
        ref_logp_lose = sequence_logprob(reference, lose_batch, PROMPT_LEN)
    # implicit-reward margin: how much more the policy favours the winner than the
    # loser, each measured relative to the reference.
    margin = beta * ((policy_logp_win - ref_logp_win) - (policy_logp_lose - ref_logp_lose))
    return -F.logsigmoid(margin).mean()


policy_dpo = copy.deepcopy(policy)
optimizer = torch.optim.AdamW(policy_dpo.parameters(), lr=config["dpo_lr"])
logp_win_history, logp_lose_history = [], []

for step in range(config["dpo_steps"]):
    batch_indices = torch.randint(0, len(winners), (config["dpo_batch"],))
    loss = dpo_loss(policy_dpo, reference, winners[batch_indices], losers[batch_indices], config["beta"])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 20 == 0:                          # track absolute log-probs (for later)
        with torch.no_grad():
            logp_win_history.append(sequence_logprob(policy_dpo, winners[:400], PROMPT_LEN).mean().item())
            logp_lose_history.append(sequence_logprob(policy_dpo, losers[:400], PROMPT_LEN).mean().item())
print(f"DPO done | final loss {loss.item():.3f}")

### Did it work?

DPO moved the model from **neutral** to **positive**: sampled reviews now use positive
adjectives far more often. Nothing scored the text during training except the fixed
preference pairs.

In [ ]:
print(f"positivity  SFT: {mean_positivity(policy):+.2f}   ->   DPO: {mean_positivity(policy_dpo):+.2f}")
print("\nsample reviews after DPO:")
for sequence in generate(policy_dpo, 6):
    print("  ", decode(sequence))

# count how often each positivity level shows up, before vs after
sft_positivity = [positivity(generate(policy, 400)[row, PROMPT_LEN:].tolist()) for row in range(400)]
dpo_positivity = [positivity(generate(policy_dpo, 400)[row, PROMPT_LEN:].tolist()) for row in range(400)]
levels = [-2, -1, 0, 1, 2]
sft_counts = [sft_positivity.count(level) for level in levels]
dpo_counts = [dpo_positivity.count(level) for level in levels]

positions = np.arange(len(levels))
bar_width = 0.4
plt.figure(figsize=(7, 4))
plt.bar(positions - bar_width / 2, sft_counts, bar_width, label="SFT (before)", color="tab:gray")
plt.bar(positions + bar_width / 2, dpo_counts, bar_width, label="DPO (after)", color="tab:green")
plt.xticks(positions, levels)
plt.xlabel("review positivity  (#positive - #negative)")
plt.ylabel("count (of 400 samples)")
plt.title("DPO shifts generation from neutral to positive")
plt.legend()
plt.tight_layout()
plt.show()

## Your LM Is Secretly a Reward Model &mdash; TASK 3

The lecture's punchline: after DPO, the policy *is* a reward model. Its **implicit
reward** costs nothing extra to read off,
$$\hat r(y) = \beta\log\frac{\pi_\theta(y)}{\pi_{\text{ref}}(y)}.$$
We check the claim two ways: (1) $\hat r$ should rank responses by their true
**positivity** (the latent thing the preferences encoded); (2) $\hat r$ should agree
with a **separately-trained Bradley--Terry reward model** on the same pairs -- exactly
the $r_\phi \leftrightarrow \hat r$ identity from the derivation. **TASK 3** is the
implicit reward itself.

In [ ]:
def implicit_reward(policy, reference, sequences, beta):
    """The reward DPO implicitly optimizes, read straight off the policy."""
    return beta * (sequence_logprob(policy, sequences, PROMPT_LEN)
                   - sequence_logprob(reference, sequences, PROMPT_LEN))


# a separately-trained Bradley-Terry reward model (same backbone, scalar head)
class RewardModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.token_emb = copy.deepcopy(backbone.token_emb)
        self.pos_emb = copy.deepcopy(backbone.pos_emb)
        self.blocks = copy.deepcopy(backbone.blocks)
        self.ln = copy.deepcopy(backbone.ln)
        self.score = nn.Linear(backbone.token_emb.embedding_dim, 1)

    def forward(self, idx):
        seq_len = idx.shape[1]
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
        x = self.token_emb(idx) + self.pos_emb(torch.arange(seq_len))
        for block in self.blocks:
            x = block(x, causal_mask)
        return self.score(self.ln(x).mean(dim=1)).squeeze(-1)


reward_model = RewardModel(policy)
optimizer = torch.optim.AdamW(reward_model.parameters(), lr=config["rm_lr"])
for step in range(config["rm_steps"]):
    batch_indices = torch.randint(0, len(winners), (config["dpo_batch"],))
    loss = -F.logsigmoid(reward_model(winners[batch_indices]) - reward_model(losers[batch_indices])).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# evaluate both rewards on fresh samples
test_samples = generate(policy, 300)
with torch.no_grad():
    r_hat = implicit_reward(policy_dpo, reference, test_samples, config["beta"]).numpy()
    r_phi = reward_model(test_samples).numpy()
gold_positivity = np.array([positivity(test_samples[row, PROMPT_LEN:].tolist())
                            for row in range(len(test_samples))])

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.scatter(gold_positivity + np.random.uniform(-0.08, 0.08, len(test_samples)), r_hat, alpha=0.5, color="tab:green")
left.set_xlabel("true positivity of the response")
left.set_ylabel(r"implicit reward  $\beta\log\pi_\theta/\pi_{ref}$")
left.set_title(f"Implicit reward tracks positivity  (rho={np.corrcoef(r_hat, gold_positivity)[0,1]:.2f})")
left.grid(alpha=0.3)
right.scatter(r_phi, r_hat, alpha=0.5, color="tab:blue")
right.set_xlabel(r"trained reward model  $r_\phi(y)$")
right.set_ylabel(r"implicit reward  $\hat r(y)$")
right.set_title(f"Implicit reward = the reward model  (rho={np.corrcoef(r_hat, r_phi)[0,1]:.2f})")
right.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("The implicit reward -- never trained as a reward model -- recovers both the true")
print("positivity ranking and the separately-trained RM. Your LM is secretly a reward model.")

## What DPO Gives Up

DPO's loss constrains only the **gap** between $y^+$ and $y^-$ -- nothing pins down
their absolute probabilities. With no on-policy sampling to anchor it, DPO often pushes
**both** the winner's and the loser's log-probability *down* (the loser much more),
leaking probability mass onto responses it never saw. RLHF's online RL loop would
notice; DPO cannot. Watch it happen:

In [ ]:
steps_axis = np.arange(len(logp_win_history)) * 20
plt.figure(figsize=(7, 4))
plt.plot(steps_axis, logp_win_history, label="winner  y+", color="tab:green", marker="o", ms=3)
plt.plot(steps_axis, logp_lose_history, label="loser  y-", color="tab:red", marker="o", ms=3)
plt.xlabel("DPO step")
plt.ylabel("policy log-prob of the response")
plt.title("DPO controls only the GAP: both log-probs fall (the loser far more)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"winner log-prob: {logp_win_history[0]:.1f} -> {logp_win_history[-1]:.1f}")
print(f"loser  log-prob: {logp_lose_history[0]:.1f} -> {logp_lose_history[-1]:.1f}")
print("Even the preferred response became less likely in absolute terms -- DPO only")
print("guaranteed the gap, not the levels. That is the price of no on-policy sampling.")

## Takeaways

- **DPO tunes a policy on preferences with one supervised loss** -- no reward model, no
  RL loop. Here it turned a neutral LM into a positive one from a fixed set of pairs.
- **The implicit reward $\hat r=\beta\log\pi_\theta/\pi_{\text{ref}}$**, read straight
  off the tuned policy, recovers both the true positivity ranking and a separately
  trained Bradley--Terry reward model ($\rho \approx 0.99$). **Your LM is secretly a
  reward model** -- the $L_{\text{RM}}$ and $L_{\text{DPO}}$ losses are the same
  $-\log\sigma(\Delta)$ shape, with $\hat r$ in place of $r_\phi$.
- **The cost of dropping the RL loop:** DPO only controls the *gap*, so it can push the
  probability of *both* responses down and drift off-distribution. This is DPO's
  version of Day 8's offline-RL failure: optimizing without on-policy coverage.
- Same preference loss shape as the RLHF lab; the reward model there is now folded into
  the policy here.

## Credits & Further Reading

- **Direct Preference Optimization** (the loss and the "secretly a reward model"
  identity) &mdash; Rafailov, Sharma, Mitchell, Ermon, Manning, Finn, *Direct
  Preference Optimization: Your Language Model is Secretly a Reward Model*, NeurIPS
  2023. [arxiv.org/abs/2305.18290](https://arxiv.org/abs/2305.18290)
- **The tiny-GPT architecture** &mdash; Andrej Karpathy, *nanoGPT*.
  [github.com/karpathy/nanoGPT](https://github.com/karpathy/nanoGPT)
- **A full DPO-from-scratch walkthrough** (on GPT-2; we shrink it to a CPU-sized model)
  &mdash; Sebastian Raschka, *LLMs-from-scratch*, ch. 7.
  [sebastianraschka.com/llms-from-scratch](https://sebastianraschka.com/llms-from-scratch/ch07/04_preference-tuning-with-dpo/)
- **The sentiment-control task this shrinks** &mdash; HuggingFace TRL, *Sentiment
  tuning of GPT-2 on IMDB*. [huggingface.co/docs/trl](https://huggingface.co/docs/trl)
- **The Bradley--Terry preference model** &mdash; Bradley & Terry, Biometrika, 1952.